In [1]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import boto3
from botocore.exceptions import NoCredentialsError
from dotenv import load_dotenv

# Carregar variáveis de ambiente do arquivo .env
load_dotenv()

# Obter as credenciais do MinIO do arquivo .env
minio_url = os.getenv('MINIO_ENDPOINT')
minio_access_key = os.getenv('MINIO_ACCESS_KEY')
minio_secret_key = os.getenv('MINIO_SECRET_KEY')
bucket_name = "bronze"

# Criar cliente MinIO
s3_client = boto3.client('s3',
                         endpoint_url=f'http://{minio_url}',
                         aws_access_key_id=minio_access_key,
                         aws_secret_access_key=minio_secret_key,
                         config=boto3.session.Config(signature_version='s3v4'))

# Carregar dados limpos do notebook eda.ipynb
df_filtered = pd.read_csv("../data/processed/dados_mesclados_limpos.csv")

# Converter o DataFrame limpo para o formato Parquet para armazenamento eficiente
tabela = pa.Table.from_pandas(df_filtered)
parquet_path = "../data/bronze/nafld1.parquet"
pq.write_table(tabela, parquet_path)

# Carregar arquivo Parquet para MinIO
try:
    s3_client.upload_file(parquet_path, bucket_name, "nafld1.parquet")
    print("Arquivo Parquet enviado para o bucket MinIO com sucesso.")
except FileNotFoundError:
    print("Arquivo não encontrado.")
except NoCredentialsError:
    print("Credenciais do MinIO não encontradas.")

Arquivo Parquet enviado para o bucket MinIO com sucesso.
